<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
LangChain und erste Agenten
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Hier trifft der Meeting- & Briefing-Agent seine ersten eigenständigen Entscheidungen — welches Tool zu einer Anfrage passt und wann eine direkte Antwort ausreicht. Das ist der Kern von **Planen**: Der Agent wählt den nächsten sinnvollen Schritt, bevor er handelt.

**Praxisanker:** Mara Vogt nutzt die Anfrage-Einordnung und passende Tools, um vor einem Meeting schnell zwischen direkter Antwort, Dokumentensuche und weiterer Prüfung zu unterscheiden.


In [1]:
#@title  🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
# Globales Tracing aus: Nur der Referenzlauf in Kapitel 6 wird erfasst.
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"] = "m02-erste-agenten"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.4.3
langchain-anthropic                      1.7.5
langchain-chroma                         1.1.0
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.6.6
langchain-google-genai                   4.4.0
langchain-ollama                         1.1.0
langchain-openai                         1.6.7
langchain-protocol                       0.0.19
langchain-text-splitters                 1.1.3
langgraph                                1.2.12
langgraph-checkpoint                     4.2.0
langgraph-checkpoint-sqlite              3.1.1
langgraph-prebuilt                       1.1.0
langgraph-sdk                            0.4.5

IP-Adresse: 35.234.41.156
Hostname: 156.41.234.

# 1 | Übersicht
---

Jetzt lernt der Meeting- & Briefing-Agent, selbst eines dieser Werkzeuge auszuwählen.

Der Weg durch M02:
1. Ein Chatmodell vorbereiten.
2. Eine Python-Funktion mit `@tool` für den Agenten bereitstellen.
3. Einen Agenten mit einem Tool ausführen.
4. Ein zweites Tool ergänzen und die Entscheidungen beobachten.
5. Den Nachrichtenverlauf und einen LangSmith-Trace lesen.

Für den ersten Agenten reichen **Modell, Tools und System-Prompt**. Abschnitt 2 führt weitere LangChain-Bausteine kompakt ein;


# 2 | LangChain 101: vom Modell zum Agenten
---


## 2.1 | Was ist LangChain?


LangChain ist ein Framework für Anwendungen mit Sprachmodellen. Es stellt Bausteine für Nachrichten, Prompts, Modellaufrufe, Tools und Agenten bereit. Diese Bausteine haben gemeinsame Aufrufmuster; so lassen sie sich zu größeren Abläufen verbinden.

Hier geht es um die LangChain-Seite: Zuerst sprechen wir ein Modell an, dann machen wir eine Funktion zum Tool und lassen einen Agenten entscheiden, wann er sie verwendet. Die [LangChain-Dokumentation zu Modellen](https://docs.langchain.com/oss/python/langchain/models) und [Agenten](https://docs.langchain.com/oss/python/langchain/agents) vertieft die beiden Bausteine.


## 2.2 | Die wichtigsten Bausteine


Ein einfacher LangChain-Ablauf liest sich von links nach rechts:


In [2]:
#@markdown   <p><font size="4" color='green'>  Verarbeitungskette</font> </br></p>

diagram = """
%%{init: {'theme':'light'}}%%
flowchart LR
    Input["Input<br/>(Message)"] --> Prompt

    subgraph Chain
        Prompt["Prompt<br/>(Template)"]
        Model["Model<br/>(LLM)"]
        Response["Antwort<br/>(AIMessage oder Text)"]
        Prompt --> Model --> Response
    end

    Response --> Result["Result"]
"""
mermaid(diagram, width=800)




| Baustein | Aufgabe | In M02 |
|---|---|---|
| **Messages** | Transportieren Anweisung, Anfrage und Antwort. | Vor und nach dem Modellaufruf |
| **Prompt-Template** | Setzt feste Anweisungen und wechselnde Werte zusammen. | Eine kleine Vorlage |
| **Chatmodell** | Verarbeitet Nachrichten und erzeugt eine Antwort oder einen Tool-Aufruf. | `llm` |
| **Chain** | Verbindet festgelegte Schritte in einer Reihenfolge. | Ein kurzer Vergleich |
| **Tool und Agent** | Tool führt eine Python-Funktion aus; Agent wählt den nächsten Schritt. | Hauptbeispiel ab Abschnitt 3 |

Chat-History, Retriever und RAG nutzen weitere Bausteine.


## 2.3 | Nachrichten und Rollen


Chatmodelle verarbeiten **Nachrichten** statt eines einzigen unmarkierten Textblocks. Die Rolle sagt, woher eine Nachricht kommt:

- **System**: Regeln und Aufgabe des Assistenten.
- **User**: Frage oder Arbeitsauftrag.
- **AI**: Antwort des Modells; sie kann auch einen Tool-Aufruf enthalten.

Eine Eingabe kann als Liste von Dictionaries mit `role` und `content` geschrieben werden. LangChain stellt außerdem Klassen wie `HumanMessage` und `AIMessage` bereit.


## 2.4 | Ein Chatmodell aufrufen


`init_chat_model(WORKER)` erstellt das Modellobjekt mit der im Kurs festgelegten Modellkennung. Dabei wird noch keine Frage gestellt. Erst `llm.invoke([...])` sendet die System- und User-Nachricht an das Modell. Die Rückgabe speichern wir als `modell_antwort`.


In [3]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(WORKER)

In [4]:
modell_antwort = llm.invoke([
    {"role": "system", "content": "Du erklärst kurz und verständlich."},
    {"role": "user", "content": "Was ist ein KI-Agent? Antworte in einem Satz."},
], config={"run_name": "m02_kap2_modell_antwort"})

mprint(f"**Antwort:** {modell_antwort.content}")

**Antwort:** Ein KI-Agent ist ein System, das selbstständig Informationen verarbeitet, Entscheidungen trifft und Aktionen ausführt, um ein bestimmtes Ziel zu erreichen.

## 2.5 | Prompts als wiederverwendbare Vorlagen


Ein Prompt enthält die Anweisung an das Modell. Wenn sich nur ein Teil der Anfrage ändert, hilft ein `ChatPromptTemplate`: Es speichert die Nachrichtenrollen und feste Formulierungen; Platzhalter wie `{thema}` werden beim Aufruf gefüllt.

`prompt_vorlage.invoke({...})` erstellt zunächst nur die fertigen Nachrichten. Ein Modellaufruf findet dabei noch nicht statt.


In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt_vorlage = ChatPromptTemplate([
    ("system", "Du erklärst technische Begriffe kurz und verständlich."),
    ("user", "Erkläre {thema} in zwei Sätzen."),
])

prompt_wert = prompt_vorlage.invoke({"thema": "Tool-Nutzung"}, config={"run_name": "m02_kap2_prompt_wert"})
for nachricht in prompt_wert.messages:
    mprint(f"**{type(nachricht).__name__}:** {nachricht.content}")


**SystemMessage:** Du erklärst technische Begriffe kurz und verständlich.

**HumanMessage:** Erkläre Tool-Nutzung in zwei Sätzen.

## 2.6 | Chain

Eine **Chain** verbindet Schritte, deren Reihenfolge vorher feststeht. Im folgenden Beispiel fließen die Werte durch `Prompt → Modell → Text-Parser`. Das Zeichen `|` verbindet diese Schritte. Mit `.invoke(...)` wird die gesamte Chain ausgeführt.

In [6]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = prompt_vorlage | llm | parser
chain_antwort = chain.invoke({"thema": "Tool-Nutzung"}, config={"run_name": "m02_kap2_chain_antwort"})

mprint(f"**Chain-Antwort:** {chain_antwort}")

**Chain-Antwort:** Tool-Nutzung bedeutet, dass ein KI-System externe Werkzeuge oder Funktionen verwenden kann, um Informationen abzurufen, Berechnungen durchzuführen oder Aufgaben auszuführen, die es allein nicht zuverlässig kann. Dadurch werden Antworten oft genauer, aktueller und nützlicher.

## 2.7 | Ausgabeformen: AIMessage und Text

Der direkte Modellaufruf aus Abschnitt 2.4 liefert ein `AIMessage`-Objekt. Sein Text steht in `.content`; zusätzlich kann es Metadaten oder Tool-Aufrufe enthalten. `StrOutputParser()` entnimmt die Textantwort. Deshalb liefert die Chain aus Abschnitt 2.6 einen einfachen Python-String.


In [7]:
mprint(f"**Direkter Modellaufruf:** `{type(modell_antwort).__name__}`")
mprint(f"**Chain mit Text-Parser ist Text:** `{isinstance(chain_antwort, str)}`")

**Direkter Modellaufruf:** `AIMessage`

**Chain mit Text-Parser ist Text:** `True`

## 2.8 | Strukturierte Ausgaben

Text ist nicht immer die passende Rückgabe. Für nachgelagerte Programmschritte kann ein Schema feste Felder verlangen, zum Beispiel `antwort` und `naechster_schritt`. LangChain bietet dafür `llm.with_structured_output(Schema)`; das Schema kann mit Pydantic definiert werden.

Für den ersten Agenten genügt hier die Unterscheidung: **Freitext** ist zum Lesen gedacht, **strukturierte Ausgabe** für klar benannte Felder.


# 3 | Von der Python-Funktion zum Tool
---


Mit `@tool` erhält eine Python-Funktion einen Namen, eine Beschreibung aus dem Docstring und ein Argument-Schema aus den Typannotationen. Der Agent kann sie dadurch einem Chatmodell anbieten und bei Bedarf ausführen.

Wir beginnen mit **einem** Tool und testen es erst ohne Agentenaufruf.


In [8]:
from langchain_core.tools import tool
from langchain.agents import create_agent

In [9]:
@tool
def themen_extrahieren(text: str) -> str:
    """Extrahiert bis zu fünf Schlüsselbegriffe aus einem kurzen Fachtext."""
    stopwoerter = {"und", "oder", "der", "die", "das", "ein", "eine", "mit", "für", "von", "im", "in"}
    woerter = [wort.strip(".,:;!?()[]").lower() for wort in text.split()]
    kandidaten = []
    for wort in woerter:
        if len(wort) >= 5 and wort not in stopwoerter and wort not in kandidaten:
            kandidaten.append(wort)
    return ", ".join(kandidaten[:5]) or "Keine Schlüsselbegriffe gefunden."


In [10]:
mprint(f"**Tool-Name:** `{themen_extrahieren.name}`")
mprint(f"**Beschreibung:** {themen_extrahieren.description}")
mprint(f"**Parameter:** `{themen_extrahieren.args}`")
direkter_test = themen_extrahieren.invoke(
    {'text': 'RAG verbindet Retrieval und Antworten.'},
    config={"run_name": "m02_kap3_tool_direkttest"},
)
mprint(f"**Direkter Test:** {direkter_test}")


**Tool-Name:** `themen_extrahieren`

**Beschreibung:** Extrahiert bis zu fünf Schlüsselbegriffe aus einem kurzen Fachtext.

**Parameter:** `{'text': {'title': 'Text', 'type': 'string'}}`

**Direkter Test:** verbindet, retrieval, antworten

**Einordnung: `llm.bind_tools()`**

Mit `llm.bind_tools([...])` können Tools direkt einem Chatmodell zur Auswahl angeboten. Die Modellantwort kann daraufhin Tool-Aufrufe enthalten; deren Ausführung und die Rückgabe der Ergebnisse an das Modell muss man bei diesem Ansatz **selbst** organisieren. LangChain verpackt ein ausgeführtes Ergebnis dabei als `ToolMessage`. `create_agent()` übernimmt diesen Ablauf.



In [11]:
llm_with_tools = llm.bind_tools([themen_extrahieren])
tool_response = llm_with_tools.invoke(
    "Extrahiere Schlüsselbegriffe aus: Retrieval verbindet Suche und Kontext.", config={"run_name": "m02_kap3_tool_response"}
)

mprint(f"**Vom Modell angeforderte Tools:** {tool_response.tool_calls}")

**Vom Modell angeforderte Tools:** [{'name': 'themen_extrahieren', 'args': {'text': 'Retrieval verbindet Suche und Kontext.'}, 'id': 'call_HWcTCJaKCCvTFMUzHtVHPATh', 'type': 'tool_call'}]

In [12]:
#@markdown   <p><font size="4" color='green'>  Kommunikationsdiagramm ohne Agent</font> </br></p>

diagram = """
%%{init: {'theme':'forest'}}%%
sequenceDiagram
    autonumber
    actor U as Benutzer
    participant C as Python-Code
    participant L as LLM
    participant T as Tool

    U->>C: Frage
    C->>L: Anfrage + Tool-Beschreibungen

    L-->>C: Tool-Call<br/>themen_extrahieren(text)

    C->>T: Tool ausführen
    T-->>C: Tool-Ergebnis

    C->>L: Tool-Ergebnis + bisheriger Kontext
    L-->>C: Finale Antwort

    C-->>U: Antwort
"""
mermaid(diagram, width=800)


# 4 | Erster Agent mit einem Tool
---


<p><font color='darkblue' size="4">
 <b>✨ Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)

`create_agent()` verbindet drei Teile:

| Parameter | Aufgabe |
|---|---|
| `model` | Das zuvor initialisierte Chatmodell. |
| `tools` | Die Funktionen, die der Agent verwenden darf. |
| `system_prompt` | Die Regel, wann und wie der Agent antworten soll. |

Der System-Prompt bleibt zunächst kurz.


In [13]:
#@markdown   <p><font size="4" color='green'>  Kommunikationsdiagramm mit Agent</font> </br></p>

diagram = """
%%{init: {'theme':'forest'}}%%
sequenceDiagram
    autonumber
    actor U as Benutzer
    participant A as Agent
    participant L as LLM
    participant T as Tool

    U->>A: Frage
    A->>L: System-Prompt + Frage + Tool-Beschreibungen

    loop Solange weitere Tool-Aufrufe erforderlich sind
        L-->>A: Tool-Call<br/>themen_extrahieren(text)
        Note right of L: Das LLM führt das Tool nicht selbst aus,<br/>es fordert den Aufruf nur an.
        A->>T: Tool ausführen
        T-->>A: Tool-Ergebnis (oder Fehlermeldung)
        A->>L: Tool-Ergebnis + bisheriger Kontext
    end

    L-->>A: Finale Antwort
    A-->>U: Antwort
"""
mermaid(diagram, width=800)


In [14]:
system_prompt = (
    "Du unterstützt Meeting- und Briefings. "
    "Nutze das Tool, wenn du Schlüsselbegriffe aus einem Fachtext bestimmen sollst. "
    "Antworte knapp und erfinde keine Rechercheergebnisse."
)

agent = create_agent(
    model=llm,
    tools=[themen_extrahieren],
    system_prompt=system_prompt,
)

mprint(f"**Agent bereit:** `{themen_extrahieren.name}` ist verfügbar.")

**Agent bereit:** `themen_extrahieren` ist verfügbar.

Die erste Anfrage verlangt genau die Fähigkeit des verfügbaren Tools. Die Eingabe besteht aus einer Liste von Nachrichten; `role: user` kennzeichnet die Nutzerfrage. Das Ergebnis enthält die Antwort und den Nachrichtenverlauf.


In [15]:
ergebnis = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extrahiere Schlüsselbegriffe aus: Retrieval Augmented Generation verbindet Suche, Kontext und Antwortgenerierung.",
    }]
}, config={"run_name": "m02_kap4_agent_ergebnis"})

mprint("## Agent-Antwort")
mprint(ergebnis["messages"][-1].content)

## Agent-Antwort

Schlüsselbegriffe: **Retrieval**, **Augmented Generation**, **Suche**, **Kontext**, **Antwortgenerierung**.

In [16]:
mprint("## Tool-Auswahl im ersten Lauf")
tool_aufrufe = [
    aufruf["name"]
    for nachricht in ergebnis["messages"]
    for aufruf in getattr(nachricht, "tool_calls", [])
]
mprint(f"**Aufgerufene Tools:** `{tool_aufrufe}`")

## Tool-Auswahl im ersten Lauf

**Aufgerufene Tools:** `['themen_extrahieren']`

# 5 | Zweites Tool und Verhalten testen
---


Jetzt kommt eine zweite Fähigkeit hinzu: eine einfache Prüfung der Themenpassung zu Dokumenten. Danach wird der Agenten mit **beiden** Tools erstellt.

In [17]:
@tool
def thema_einordnen(thema: str) -> str:
    """Schätzt ein, ob ein Thema zu den verfügbaren Meeting- und Projektdokumenten passt."""
    bekannte_themen = {
        "rag": "RAG ist in den verfügbaren Meeting- und Projektdokumenten vertreten.",
        "retrieval": "Retrieval ist in den verfügbaren Meeting- und Projektdokumenten vertreten.",
        "evaluation": "Evaluation ist in den verfügbaren Meeting- und Projektdokumenten vertreten.",
        "agent": "Agenten sind in den verfügbaren Meeting- und Projektdokumenten vertreten.",
        "langgraph": "LangGraph ist in den verfügbaren Meeting- und Projektdokumenten voraussichtlich nur indirekt vertreten.",
    }
    thema_normalisiert = thema.lower()
    for schluessel, antwort in bekannte_themen.items():
        if schluessel in thema_normalisiert:
            return antwort
    return "Keine sichere Themenpassung zu den Meeting- und Projektdokumenten erkennbar."


In [18]:
system_prompt = (
    "Du unterstützt Meeting- und Briefings. "
    "Nutze themen_extrahieren für Schlüsselbegriffe und thema_einordnen für Fragen zur Themenpassung zu Meeting- und Projektdokumenten. "
    "Wenn keine dieser beiden Aufgaben gefragt ist, nutze kein Tool und antworte direkt. "
    "Antworte knapp und erfinde keine Rechercheergebnisse."
)

agent = create_agent(
    model=llm,
    tools=[themen_extrahieren, thema_einordnen],
    system_prompt=system_prompt,
)

mprint("**Agent bereit** mit `themen_extrahieren` und `thema_einordnen`.")


**Agent bereit** mit `themen_extrahieren` und `thema_einordnen`.

In [19]:
test_fragen = [
    {
        "name": "Schlüsselbegriffe",
        "frage": "Welche Schlüsselbegriffe stecken in: Agentic RAG kombiniert Planung, Retrieval und Evaluation.",
        "erwartete_tools": {"themen_extrahieren"},
    },
    {
        "name": "Themenpassung zu Dokumenten",
        "frage": "Gibt es zu Evaluation passende Meeting- oder Projektdokumente?",
        "erwartete_tools": {"thema_einordnen"},
    },
    {
        "name": "Direkte Antwort",
        "frage": "Was ist Weihnachten.",
        "erwartete_tools": set(),
    },
    {
        "name": "Beide Tools",
        "frage": "Prüfe, ob es zu RAG passende Meeting- oder Projektdokumente gibt, und extrahiere Begriffe aus: RAG nutzt Retrieval für fundierte Antworten.",
        "erwartete_tools": {"thema_einordnen", "themen_extrahieren"},
    },
]


In [20]:
for _i, (testfall) in enumerate(test_fragen, 1):
    result = agent.invoke({
        "messages": [{"role": "user", "content": testfall["frage"]}]
    }, config={"run_name": f"m02_kap5_agent_result_{_i}"})

    tool_calls = [
        call
        for message in result["messages"]
        for call in getattr(message, "tool_calls", [])
    ]
    tatsaechliche_tools = {call["name"] for call in tool_calls}

    tool_messages = [
        message
        for message in result["messages"]
        if type(message).__name__ == "ToolMessage"
    ]

    status = (
        "OK"
        if tatsaechliche_tools == testfall["erwartete_tools"]
        else "ABWEICHUNG"
    )

    mprint(f"\n---\n**{testfall['name']}**")
    mprint(f"**Frage:** {testfall['frage']}")
    mprint(f"**Erwartete Tools:** `{sorted(testfall['erwartete_tools'])}`")
    mprint(f"**Tatsächliche Tool-Calls:** `{tool_calls}`")
    mprint(f"**Tool-Namen:** `{sorted(tatsaechliche_tools)}`")

    if tool_messages:
        for message in tool_messages:
            mprint(
                f"**ToolMessage** `{message.name or message.tool_call_id}`: "
                f"{message.content}"
            )
    else:
        mprint("**ToolMessage:** keine")

    mprint(f"**Antwort:** {result['messages'][-1].content}")
    mprint(f"**Status:** **{status}**")



---
**Schlüsselbegriffe**

**Frage:** Welche Schlüsselbegriffe stecken in: Agentic RAG kombiniert Planung, Retrieval und Evaluation.

**Erwartete Tools:** `['themen_extrahieren']`

**Tatsächliche Tool-Calls:** `[{'name': 'themen_extrahieren', 'args': {'text': 'Agentic RAG kombiniert Planung, Retrieval und Evaluation.'}, 'id': 'call_FpgyA2P00SYem3gXiMzHUeAy', 'type': 'tool_call'}]`

**Tool-Namen:** `['themen_extrahieren']`

...

**Tool-Namen:** `['thema_einordnen', 'themen_extrahieren']`

**ToolMessage** `thema_einordnen`: RAG ist in den verfügbaren Meeting- und Projektdokumenten vertreten.

**ToolMessage** `themen_extrahieren`: nutzt, retrieval, fundierte, antworten

**Antwort:** RAG kommt im Korpus vor.

Begriffe: nutz t, Retrieval, fundierte, Antworten

**Status:** **OK**

# 6 | Nachrichten und Entscheidungen verstehen
---

Im gespeicherten ersten Lauf `ergebnis` stehen die einzelnen Schritte unter `messages`:

- **`HumanMessage`**: die Nutzerfrage.
- **`AIMessage`**: eine Tool-Entscheidung oder die abschließende Antwort des Modells.
- **`ToolMessage`**: das Ergebnis eines ausgeführten Tools.

Bei einer festen Python-Abfolge wären die Aufrufe vorher festgelegt. Hier entscheidet der Agent anhand der Anfrage, welches Tool er nutzt.


In [21]:
nachrichten = ergebnis["messages"]
mprint(f"**Nachrichten im ersten Lauf:** {len(nachrichten)}")
for nachricht in nachrichten:
    typ = type(nachricht).__name__
    tool_namen = [aufruf["name"] for aufruf in getattr(nachricht, "tool_calls", [])]
    inhalt = str(nachricht.content)[:160] or "(kein Text)"
    mprint(f"- **{typ}**: {inhalt} – Tool-Aufrufe: `{tool_namen}`")


**Nachrichten im ersten Lauf:** 4

- **HumanMessage**: Extrahiere Schlüsselbegriffe aus: Retrieval Augmented Generation verbindet Suche, Kontext und Antwortgenerierung. – Tool-Aufrufe: `[]`

- **AIMessage**: (kein Text) – Tool-Aufrufe: `['themen_extrahieren']`

- **ToolMessage**: retrieval, augmented, generation, verbindet, suche – Tool-Aufrufe: `[]`

- **AIMessage**: Schlüsselbegriffe: **Retrieval**, **Augmented Generation**, **Suche**, **Kontext**, **Antwortgenerierung**. – Tool-Aufrufe: `[]`

# 7 | LangSmith: Agent-Trace
---


Ein Trace zeigt die sichtbaren Schritte eines Agentenlaufs: Modellentscheidung, Tool-Aufruf mit Parametern, Tool-Ergebnis und Antwort. Er ergänzt den Nachrichtenverlauf um eine gut lesbare Ablaufansicht.

Für diesen Referenzlauf wird LangSmith einmal gezielt aktiviert. Versteckte Modellgedanken sind dabei nicht sichtbar.


In [22]:
setup_api_keys(["LANGSMITH_API_KEY"], create_globals=False)
from langsmith import tracing_context

run_cfg = {
    "run_name": "m02_kap7_agenttrace",
    "tags": ["m02", "agent", "langsmith", "meeting-research-briefing"],
}

with tracing_context(enabled=True):
    trace_result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Gibt es zu Retrieval passende Meeting- oder Projektdokumente, und welche Schlüsselbegriffe enthält: Retrieval verbessert Antworten durch dokumentnahen Kontext?",
        }]
    }, config=run_cfg)

mprint("## Trace-Ergebnis")
mprint(trace_result["messages"][-1].content)


✓ LANGSMITH_API_KEY erfolgreich gesetzt


## Trace-Ergebnis

Ja, Retrieval ist in den verfügbaren Meeting- und Projektdokumenten vertreten.

Schlüsselbegriffe:
- retrieval
- verbessert
- antworten
- durch
- dokumentnahen

In [23]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("m02-erste-agenten", limit=3, show_steps=True)

## LangSmith Trace — `M02-Erste-Agenten`

| Run | Status | Dauer | Child-Runs |
|-----|--------|-------|------------|
| `M02_Kap7_AgentTrace` | ⏳ pending | — | 4 |


### Steps — letzter Run: `M02_Kap7_AgentTrace`

| # | Typ | Name | Status | Dauer |
|---|-----|------|--------|-------|
| 1 | `chain` | `model` | ✅ | 1.4s |
| 2 | `chain` | `tools` | ✅ | 0.0s |
| ↳ | `tool` | `thema_einordnen` | ✅ | 0.0s |
| 3 | `chain` | `tools` | ✅ | 0.0s |
| ↳ | `tool` | `themen_extrahieren` | ✅ | 0.0s |
| 4 | `chain` | `model` | ✅ | 1.1s |

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.




<p><font color='black' size="5">
Erster eigener Agent
</font></p>

Einen ersten Agenten für den Meeting-Briefing-Agent erstellen. Ein Tool wird definiert, in `create_agent()` eingebunden und mit einer Anfrage getestet.

**Grundlagen**
1. Ein eigenes Briefing-Tool mit dem `@tool`-Decorator definieren, zum Beispiel `briefing_signal_extrahieren(text: str)`.
2. Einen Agenten mit `create_agent()` und diesem Tool erstellen.
3. Eine einfache Testfrage stellen, bei der das Tool nützlich ist, und die Ausgabe zeigen.

Den Agenten in der Variable `mein_agent` speichern.


In [24]:
from genai_lib.model_config import WORKER
# Grundlagen: Ein Tool + Agent
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool

# ...

**Aufbau**
1. Zwei Briefing-Tools definieren: eines für Textsignale, eines für Themenpassung zu Dokumenten.
2. Einen System-Prompt schreiben, der die Rolle des Agenten klar beschreibt.
3. Den Agenten mit beiden Tools und dem eigenen System-Prompt erstellen.
4. Drei Testfragen stellen - davon mindestens 1 Frage, bei der kein Tool nötig ist.
5. Kurz notieren, wann Tools genutzt werden und wann nicht.

Agenten in `mein_agent`, Tools in `meine_tools` speichern.


In [25]:
# Aufbau: Zwei Tools + System-Prompt + drei Testfragen
@tool
def count_briefing_signals(text: str) -> str:
    " ... "
    pass

@tool
def thema_einordnen_exercise(thema: str) -> str:
    " ... "
    pass

# ...

**Vertiefung**
1. Ein drittes Briefing-Tool hinzufügen.
2. 2 Anfragen stellen, bei denen der Agent selbst entscheidet, welches der 3 Tools passt.
3. Den System-Prompt anpassen, zum Beispiel strenger oder ausführlicher, und in 2-3 Sätzen beschreiben, wie sich das Verhalten ändert.
4. LangSmith-Tracing aktivieren und für einen Agenten-Lauf beschreiben, welche ReAct-Schritte im Trace sichtbar sind.

Die Beobachtung zum System-Prompt in `prompt_beobachtung` (String) speichern.


In [26]:
# Vertiefung: Drittes Tool + LangSmith
@tool
def out_of_scope_notice(frage: str) -> str:
    " ... "



# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Einsteiger LangChain](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-langchain.html)
- [LangChain](https://ralf-42.github.io/Agenten/05-frameworks/langchain.html)
- [Agenten-Architekturen](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/agent-architekturen.html)
- [Tool Use & Function Calling](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/tool-use-function-calling.html)

